# Notebook 01n - PPO baseline on `MiniGrid-KeyCorridorS3R3-v0`

Train the PPO baseline for KeyCorridor using `config/envs/keycorridors3r3.yaml`.
The PPO hyperparameters are mapped from RL Baselines3 Zoo's KeyCorridorS3R1 entry and given a larger frame budget for S3R3. This repo keeps fully observable MiniGrid image observations so the
later PPO-CF oracle is operating on a Markov state.


---
## Knobs


In [ ]:
# ----------------------------------------------------------------------------
# EDIT ME
# ----------------------------------------------------------------------------
ENV_CONFIG    = "keycorridors3r3"
SEEDS         = None          # None -> use the config's seeds
FORCE_RETRAIN = True

OVERRIDES = {
    # YAML now contains the working fixed-layout lr=5e-4 recipe.
    # Quick smoke:
    # "ppo.total_timesteps": 100_000,
    # "run.seeds": (0,),
}
# ----------------------------------------------------------------------------


## 0. Setup


In [2]:
%load_ext autoreload
%autoreload 2

import sys, pathlib, time

ROOT = pathlib.Path.cwd()
if not (ROOT / "config").is_dir():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from config import make_config, seed_dir, RUNS_DIR, FIGURES_DIR
from dataio import load_trajectories, validate, load_checkpoint, list_checkpoints
from envs.env_pool import make_env
from utils.logging import read_scalars
from utils.plotting import plot_learning_curves, plot_episode_returns, plot_grid, savefig
from scripts.train import run_seeds

cfg = make_config(ENV_CONFIG, **OVERRIDES)
if SEEDS is not None:
    cfg = make_config(ENV_CONFIG, **{**OVERRIDES, "run.seeds": tuple(SEEDS)})
SEEDS = tuple(cfg.run.seeds)
RUN_NAME = cfg.run.run_name
FIG = FIGURES_DIR / f"nb01_keycorridors3r3_{RUN_NAME}"
FIG.mkdir(parents=True, exist_ok=True)

ACTION_NAMES = ["left", "right", "forward", "pickup", "drop", "toggle", "done"]
SUBGOAL_NAMES = ("key picked", "locked door open")
print(cfg.summary())
print()
print(f"run: {RUN_NAME}   seeds: {list(SEEDS)}")


config             keycorridors3r3_cf
env                MiniGrid-KeyCorridorS3R3-v0  (n_envs=8, obs_norm=image)
layouts            1 fixed (cycle)
total_timesteps    2,000,000  per seed
rollout batch      1024  (8 envs x 128 steps)
updates            1,953
minibatch size     64  x 10 epochs
gamma / lambda     0.99 / 0.95
lr                 0.0005  (anneal=False)
encoder            cnn  (shared=True)
entropy            fixed, coef 0.1
prob floor         0.03 -> 0.001  (0 = off)
adv norm           batch  (min_std 1e-06)
policy gradient    cf_all_action  alpha_gae=1.0 alpha_cf=0.25, Q_g horizon=96 x2 rollouts, subsample=4%, restore=fast


reward shaping     off
warm start         none
seeds              [0]
checkpoints at     ['10%', '30%', '50%', '75%', '100%']

run: keycorridors3r3_fixed0_lr5e4   seeds: [0]


## 1. Train


In [3]:
already = all((seed_dir(RUN_NAME, s) / "scalars.csv").exists() for s in SEEDS)

if already and not FORCE_RETRAIN:
    print(f"found existing run at {RUNS_DIR / RUN_NAME} -- skipping training")
else:
    t0 = time.time()
    outputs = run_seeds(cfg)
    print()
    print(f"training finished in {(time.time() - t0) / 60:.1f} min")



=== seed 0 ====================================================


/opt/homebrew/anaconda3/envs/workbench/lib/python3.10/site-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


  oracle check: replay exact over 96 transitions (reward err 0, state err 0); centering max 9.82e-09
  upd    10/1953  step   10,240  ret   0.010  succ 0.03  sg1 0.12 sg2 0.03  ent 1.872  advstd 6.40e-03  ev  0.402  166 sps
  upd    20/1953  step   20,480  ret   0.004  succ 0.01  sg1 0.14 sg2 0.03  ent 1.883  advstd 9.79e-03  ev  0.630  172 sps
  upd    30/1953  step   30,720  ret   0.000  succ 0.00  sg1 0.21 sg2 0.06  ent 1.914  advstd 6.14e-03  ev  0.783  172 sps
  upd    40/1953  step   40,960  ret   0.000  succ 0.00  sg1 0.24 sg2 0.07  ent 1.847  advstd 6.03e-03  ev -0.266  172 sps
  upd    50/1953  step   51,200  ret   0.000  succ 0.00  sg1 0.25 sg2 0.06  ent 1.891  advstd 8.13e-03  ev -0.255  171 sps
  upd    60/1953  step   61,440  ret   0.000  succ 0.00  sg1 0.44 sg2 0.06  ent 1.891  advstd 7.43e-03  ev -0.576  171 sps
  upd    70/1953  step   71,680  ret   0.000  succ 0.00  sg1 0.49 sg2 0.06  ent 1.857  advstd 4.76e-03  ev -0.221  171 sps
  upd    80/1953  step   81,920  ret  

KeyboardInterrupt: 

## 2. Load Outputs


In [ ]:
scalars = {}
episodes = {}
for s in SEEDS:
    sd = seed_dir(RUN_NAME, s)
    if (sd / "scalars.csv").exists():
        scalars[s] = read_scalars(sd / "scalars.csv")
    if (sd / "episodes.csv").exists():
        episodes[s] = pd.read_csv(sd / "episodes.csv")

summary = []
for s, d in scalars.items():
    row = {
        "seed": s,
        "steps": int(d["global_step"].iloc[-1]),
        "final_return_100": round(float(d["mean_return_100"].iloc[-1]), 3),
        "final_success_100": round(float(d["success_rate_100"].iloc[-1]), 3),
        "best_success_100": round(float(d["success_rate_100"].max()), 3),
        "final_sg1": round(float(d["subgoal1_rate_100"].iloc[-1]), 3) if "subgoal1_rate_100" in d else None,
        "final_sg2": round(float(d["subgoal2_rate_100"].iloc[-1]), 3) if "subgoal2_rate_100" in d else None,
        "median_entropy": round(float(d["entropy"].median()), 3),
        "median_ev": round(float(d["explained_variance"].median()), 3),
    }
    summary.append(row)
display(pd.DataFrame(summary))


## 3. Return And Subgoals


In [ ]:
if scalars:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    plot_learning_curves(scalars, y="mean_return_100", ylabel="return", title="Return", ax=axes[0])
    plot_learning_curves(scalars, y="success_rate_100", ylabel="success", title="Success", hline=0.8, ax=axes[1])
    if "subgoal1_rate_100" in next(iter(scalars.values())).columns:
        plot_learning_curves(scalars, y="subgoal1_rate_100", ylabel=SUBGOAL_NAMES[0], title=SUBGOAL_NAMES[0], ax=axes[2])
        plot_learning_curves(scalars, y="subgoal2_rate_100", ylabel=SUBGOAL_NAMES[1], title=SUBGOAL_NAMES[1], ax=axes[3])
    for ax in axes[1:]:
        ax.set_ylim(-0.05, 1.05)
    fig.tight_layout()
    savefig(fig, FIG / "learning.png")
    plt.show()


## 4. Optimisation Diagnostics


In [ ]:
if scalars:
    keys = [k for k in ["loss", "policy_loss", "value_loss", "approx_kl", "clipfrac", "adv_std", "entropy", "explained_variance"]
            if k in next(iter(scalars.values())).columns]
    fig = plot_grid(scalars, keys, ncols=4, figsize=(16, 7))
    savefig(fig, FIG / "optimisation.png")
    plt.show()


## 5. Action Diagnostics


In [ ]:
action_rows = []
for s in SEEDS:
    path = seed_dir(RUN_NAME, s) / "trajectories.npz"
    if not path.exists():
        continue
    traj = load_trajectories(path)
    acts = traj.action.astype(int)
    probs = traj.probs.astype(float)
    for a, name in enumerate(ACTION_NAMES):
        action_rows.append({
            "seed": s,
            "action": name,
            "sample_frac": float((acts == a).mean()),
            "mean_pi": float(probs[:, a].mean()),
            "p05_pi": float(np.quantile(probs[:, a], 0.05)),
            "p95_pi": float(np.quantile(probs[:, a], 0.95)),
        })
action_df = pd.DataFrame(action_rows)
display(action_df)

if not action_df.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    piv = action_df.pivot(index="action", columns="seed", values="mean_pi").loc[ACTION_NAMES]
    piv.plot(kind="bar", ax=axes[0], legend=True)
    axes[0].set_title("Mean policy probability")
    axes[0].set_ylabel("mean pi(a|s)")
    axes[0].spines[["top", "right"]].set_visible(False)
    piv2 = action_df.pivot(index="action", columns="seed", values="sample_frac").loc[ACTION_NAMES]
    piv2.plot(kind="bar", ax=axes[1], legend=False)
    axes[1].set_title("Sampled action fraction")
    axes[1].set_ylabel("fraction")
    axes[1].spines[["top", "right"]].set_visible(False)
    fig.tight_layout()
    savefig(fig, FIG / "actions.png")
    plt.show()


## 6. Checkpoints And Greedy Evaluation


In [ ]:
def greedy_eval(ck, n_episodes=50, seed0=90_000):
    env = make_env(cfg.env.env_id, cfg.env.max_episode_steps, fully_observable=cfg.env.fully_observable)
    returns = []
    successes = []
    lengths = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed0 + ep)
        done = False
        ret = 0.0
        t = 0
        while not done:
            x = np.asarray(obs, dtype=np.float32).reshape(1, -1)
            p = ck.probs(x)[0]
            a = int(np.argmax(p))
            obs, r, term, trunc, _ = env.step(a)
            ret += float(r)
            t += 1
            done = bool(term or trunc)
        returns.append(ret)
        successes.append(bool(term and ret > 0))
        lengths.append(t)
    env.close()
    return {
        "greedy_return": float(np.mean(returns)),
        "greedy_success": float(np.mean(successes)),
        "greedy_len": float(np.mean(lengths)),
    }

rows = []
for s in SEEDS:
    ckpts = list_checkpoints(seed_dir(RUN_NAME, s) / "checkpoints")
    if not ckpts:
        continue
    for p in ckpts:
        ck = load_checkpoint(p)
        rows.append({"seed": s, "fraction": ck.fraction, "global_step": ck.global_step, **greedy_eval(ck)})

evaldf = pd.DataFrame(rows)
display(evaldf)


## 7. Trajectory Validation


In [ ]:
all_clean = True
for s in SEEDS:
    path = seed_dir(RUN_NAME, s) / "trajectories.npz"
    if not path.exists():
        print(f"seed {s}: no trajectories at {path}")
        all_clean = False
        continue
    print(f"seed {s}")
    problems = validate(load_trajectories(path), n_actions=len(ACTION_NAMES), verbose=True)
    if problems:
        all_clean = False
        for p in problems:
            print("  -", p)
print()
print("trajectory validation:", "PASS" if all_clean else "CHECK")


## 8. Gate 1 Verdict


In [ ]:
checks = []
if scalars:
    best_success = max(float(d["success_rate_100"].max()) for d in scalars.values())
    checks.append(("success_rate_100 ever >= 0.80", best_success >= 0.80, best_success))
if "evaldf" in globals() and not evaldf.empty:
    best_greedy = float(evaldf["greedy_success"].max())
    checks.append(("greedy_success ever >= 0.80", best_greedy >= 0.80, best_greedy))
if "action_df" in globals() and not action_df.empty:
    low_required = action_df[action_df["action"].isin(["pickup", "toggle"])] ["mean_pi"].min()
    checks.append(("pickup/toggle mean_pi not collapsed", float(low_required) > 0.005, float(low_required)))
checks.append(("trajectory dataset valid", all_clean, all_clean))

for name, ok, value in checks:
    print(f"{name:40s} {'PASS' if ok else 'CHECK':5s}  {value}")
